# WRI Aqueduct — Clean 13-Score Facility Pipeline

**Purpose:** Build a new, self-contained pipeline: WRI polygons → 13 standard indicators → facility spatial join → raw and normalized facility datasets.

No ELP features are selected, analyzed, or exported. This notebook is independent of the previous notebooks.

**Inputs required:** (1) a facility CSV with `id`, `lat`, `lon`; (2) either a WRI polygon GeoParquet or the Aqueduct geodatabase with a `baseline_annual` layer.

**Normalization decision:** Raw WRI scores are retained; a second table uses **fixed-scale normalization, score / 5**, yielding [0,1]. This preserves the original published 0–5 meaning and avoids data-derived min/max leakage. For standardization (z-scores) in a model, fit a scaler on the training split only.


## 1. Configuration

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd

ROOT = Path.cwd()

assert (ROOT / "Data" / "processed").is_dir(), (
    f"Run this notebook from the ATLAS repository root: {ROOT}"
)

# Historical source used to reproduce the original spatial-join experiment.
FACILITY_CSV = ROOT / "Data" / "processed" / "im3_eia_combined.csv"

# Current canonical facility source for updated ATLAS experiments.
ATLAS_CSV = ROOT / "Data" / "processed" / "im3_atlas_combined.csv"

# Choose ONE source: GeoParquet (preferred if already cleaned) or original geodatabase.
WRI_PARQUET = ROOT / "Data" / "processed" / "wri_us_cleaned.parquet"
WRI_GDB = None  # e.g. ROOT / "Data" / "raw" / "Aqueduct40.gdb"
WRI_LAYER = "baseline_annual"

OUT = ROOT / "Data" / "processed"
assert OUT.is_dir(), f"Missing processed data directory: {OUT}"

SCORES = [
    "bws_score", "bwd_score", "iav_score", "sev_score",
    "gtd_score", "rfr_score", "cfr_score", "drr_score",
    "ucw_score", "cep_score", "udw_score", "usa_score",
    "rri_score",
]
assert len(SCORES) == 13

# Pipeline execution settings
WRI_MODE = "reuse"  # "reuse" or "regenerate"
WRITE_OUTPUTS = False

assert WRI_MODE in {"reuse", "regenerate"}

## 2. Load facilities and validate coordinates

In [ ]:
if WRI_MODE == "regenerate":
    # Existing cell contents indented here    
    assert FACILITY_CSV.exists(), f"Missing facility file: {FACILITY_CSV}"
    fac = pd.read_csv(FACILITY_CSV, dtype={"id": "string"})
    required = {"id", "lat", "lon"}
    assert required.issubset(fac.columns), f"Missing facility columns: {required - set(fac.columns)}"
    assert fac["id"].notna().all() and fac["id"].is_unique
    fac["lat"] = pd.to_numeric(fac["lat"], errors="coerce")
    fac["lon"] = pd.to_numeric(fac["lon"], errors="coerce")
    assert fac["lat"].between(-90, 90).all() and fac["lon"].between(-180, 180).all(), "Invalid/missing coordinates"
    points = gpd.GeoDataFrame(
        fac[["id"]].copy(),
        geometry=gpd.points_from_xy(fac["lon"], fac["lat"]),
        crs="EPSG:4326",
    )
    print(f"Facilities: {len(fac):,}; unique IDs: {fac.id.nunique():,}")


Facilities: 1,147; unique IDs: 1,147


## 3. Load WRI polygon layer (13 scores only)

In [ ]:
if WRI_MODE == "regenerate":
    # Existing cell contents indented here
    if WRI_PARQUET is not None and Path(WRI_PARQUET).is_file():
        wri = gpd.read_parquet(WRI_PARQUET)
        print("Loaded WRI GeoParquet:", WRI_PARQUET)
    elif WRI_GDB is not None and Path(WRI_GDB).exists():
        wri = gpd.read_file(WRI_GDB, layer=WRI_LAYER)
        print("Loaded WRI geodatabase:", WRI_GDB)
    else:
        raise FileNotFoundError("Set WRI_PARQUET to an existing file, or WRI_GDB to the source geodatabase.")

    needed = set(SCORES + ["pfaf_id", "geometry"])
    assert needed.issubset(wri.columns), f"Missing WRI columns: {needed - set(wri.columns)}"
    assert wri.crs is not None, "WRI polygons have no CRS"
    if "gid_0" in wri.columns:
        wri = wri.loc[wri["gid_0"].astype("string").eq("USA")].copy()
    wri = wri[["pfaf_id"] + SCORES + ["geometry"]].copy()
    wri = wri.loc[wri.geometry.notna() & ~wri.geometry.is_empty].copy()
    wri = wri.to_crs(points.crs)
    print(f"WRI polygons: {len(wri):,}; CRS: {wri.crs}")


Loaded WRI GeoParquet: /mnt/c/Users/attho/OneDrive/Desktop/College/ACM ATLAS/ATLAS/Data/processed/wri_us_cleaned.parquet
WRI polygons: 3,433; CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction

## 4. Clean score values and inspect missingness

In [ ]:
if WRI_MODE == "regenerate":
    # Existing cell contents indented here
    for col in SCORES:
        wri[col] = pd.to_numeric(wri[col], errors="coerce")
        wri[col] = wri[col].replace([-9999, 9999], np.nan)
        invalid = wri[col].notna() & ~wri[col].between(0, 5)
        assert not invalid.any(), f"{col}: {invalid.sum()} values outside 0–5"

    polygon_quality = pd.DataFrame({
        "missing_n": wri[SCORES].isna().sum(),
        "missing_pct": (100 * wri[SCORES].isna().mean()).round(2),
        "unique_nonmissing": wri[SCORES].nunique(dropna=True),
    })
    display(polygon_quality)


,missing_n,missing_pct,unique_nonmissing
bws_score,127,3.70,474
bwd_score,127,3.70,1027
iav_score,127,3.70,1215
sev_score,127,3.70,1241
gtd_score,2340,68.16,27
rfr_score,102,2.97,1081
cfr_score,102,2.97,125
drr_score,235,6.85,1179
ucw_score,0,0.00,1
cep_score,116,3.38,691


## 5. Facility-level spatial join

Use `within` to attach the WRI polygon to each facility point. Check unmatched facilities and polygon overlaps explicitly; **never silently discard duplicates or fill missing risk values with zero**. A facility exactly on a polygon boundary may not match with `within`; investigate those cases individually.


In [ ]:
if WRI_MODE == "regenerate":
    # Existing cell contents indented here    
    joined = gpd.sjoin(
        points,
        wri,
        how="left",
        predicate="within",
    )
    match_counts = joined.groupby("id", dropna=False).size()
    multi_ids = match_counts[match_counts > 1]
    unmatched = joined["index_right"].isna()
    print(f"Input facilities: {len(points):,}")
    print(f"Joined rows: {len(joined):,}")
    print(f"Unmatched facility IDs: {joined.loc[unmatched, 'id'].nunique():,}")
    print(f"Multiple-match facility IDs: {len(multi_ids):,}")
    if len(multi_ids):
        display(joined.loc[joined["id"].isin(multi_ids.index), ["id", "pfaf_id", "index_right"]].head(20))
    assert not len(multi_ids), "Overlapping polygon matches: resolve before exporting"
    assert not unmatched.any(), "Some facility points are unmatched: inspect boundaries/coordinates before exporting"
    assert len(joined) == len(fac) and joined["id"].is_unique


Input facilities: 1,147
Joined rows: 1,147
Unmatched facility IDs: 0
Multiple-match facility IDs: 0


## 6. Facility-level quality checks

In [ ]:
if WRI_MODE == "regenerate":
    # Existing cell contents indented here
    raw_features = pd.DataFrame(joined.drop(columns=["geometry", "index_right"]))[
        ["id", "pfaf_id"] + SCORES
    ].copy()
    quality = pd.DataFrame({
        "missing_n": raw_features[SCORES].isna().sum(),
        "missing_pct": (100 * raw_features[SCORES].isna().mean()).round(2),
        "unique_nonmissing": raw_features[SCORES].nunique(dropna=True),
        "min": raw_features[SCORES].min(),
        "max": raw_features[SCORES].max(),
    })
    display(quality)
    print("Constant indicators:", quality.index[quality["unique_nonmissing"] <= 1].tolist())
    print("Important: constant/high-missing indicators are flagged, not automatically removed.")


,missing_n,missing_pct,unique_nonmissing,min,max
bws_score,0,0.00,101,0.000000,5.000000
bwd_score,0,0.00,132,0.000516,5.000000
iav_score,0,0.00,139,0.439879,4.740104
sev_score,0,0.00,139,0.141309,2.915418
gtd_score,864,75.33,12,1.064448,3.297948
rfr_score,0,0.00,139,0.000122,3.406840
cfr_score,0,0.00,26,0.000000,2.697086
drr_score,0,0.00,139,1.010861,2.924859
ucw_score,0,0.00,1,0.880683,0.880683
cep_score,0,0.00,92,0.820119,4.290276


Constant indicators: ['ucw_score', 'rri_score']
Important: constant/high-missing indicators are flagged, not automatically removed.


## 7. Join to full facility table

Use a one-to-one ID join and verify that the original facility fields are unchanged. The final raw table contains **exactly 13 WRI score columns**. Existing score columns in the facility input are treated as an error rather than silently duplicated.


In [ ]:
if WRI_MODE == "regenerate":
    # Existing cell contents indented here
    overlap = set(SCORES + ["pfaf_id"]) & set(fac.columns)
    assert not overlap, f"Facility input already has WRI columns: {sorted(overlap)}"
    combined_raw = fac.merge(raw_features, on="id", how="left", validate="one_to_one", sort=False)
    assert len(combined_raw) == len(fac)
    assert combined_raw["id"].is_unique
    assert combined_raw[fac.columns].equals(fac)
    assert len([c for c in combined_raw.columns if c.endswith("_score")]) == 13
    print("Combined raw shape:", combined_raw.shape)
    display(combined_raw[["id", "pfaf_id"] + SCORES].head())


Combined raw shape: (1147, 64)


,id,pfaf_id,bws_score,bwd_score,iav_score,sev_score,gtd_score,rfr_score,cfr_score,drr_score,ucw_score,cep_score,udw_score,usa_score,rri_score
0,0002744301,731504,3.874239,1.728125,1.593892,0.905566,NaN,0.161611,0.0,1.720637,0.880683,4.037691,0.186522,0.0,0.88
1,0007805491,742678,2.834326,1.170674,1.012947,0.623887,NaN,0.106682,0.0,2.509289,0.880683,2.934937,0.273142,0.0,0.88
2,0009474864,732409,2.942697,1.164728,1.513840,0.295139,NaN,1.014280,0.0,1.975714,0.880683,1.847858,1.254066,0.0,0.88
3,0013924557,742843,0.612164,0.322639,3.525262,1.184212,NaN,0.320215,0.0,2.909051,0.880683,2.934831,1.175798,0.0,0.88
4,0014593270,732408,3.638327,1.419415,2.502750,0.332043,NaN,1.596326,0.0,1.996600,0.880683,1.957676,1.032001,0.0,0.88


## 8. Fixed 0–1 normalization

Each standard WRI score is defined on a 0–5 scale. Dividing by 5 yields a **fixed, interpretable 0–1 transformation** (not sample min-max scaling). Missing values remain missing. Normalized columns use the suffix `_norm` and raw scores are retained in a separate file.

**Modeling note:** For models that benefit from z-scoring, split into train/validation/test first, then fit `StandardScaler` (and any imputer) on training data only. Avoid fitting preprocessing on the complete dataset.


In [ ]:
if WRI_MODE == "regenerate":
    # Existing cell contents indented here
    combined_norm = combined_raw.drop(columns=SCORES).copy()
    for col in SCORES:
        combined_norm[col.removesuffix("_score") + "_norm"] = combined_raw[col] / 5.0

    norm_cols = [c.removesuffix("_score") + "_norm" for c in SCORES]
    assert len(norm_cols) == 13
    assert all(combined_norm[c].dropna().between(0, 1).all() for c in norm_cols)
    assert not any(c in combined_norm.columns for c in SCORES)
    display(combined_norm[["id", "pfaf_id"] + norm_cols].head())


,id,pfaf_id,bws_norm,bwd_norm,iav_norm,sev_norm,gtd_norm,rfr_norm,cfr_norm,drr_norm,ucw_norm,cep_norm,udw_norm,usa_norm,rri_norm
0,0002744301,731504,0.774848,0.345625,0.318778,0.181113,NaN,0.032322,0.0,0.344127,0.176137,0.807538,0.037304,0.0,0.176
1,0007805491,742678,0.566865,0.234135,0.202589,0.124777,NaN,0.021336,0.0,0.501858,0.176137,0.586987,0.054628,0.0,0.176
2,0009474864,732409,0.588539,0.232946,0.302768,0.059028,NaN,0.202856,0.0,0.395143,0.176137,0.369572,0.250813,0.0,0.176
3,0013924557,742843,0.122433,0.064528,0.705052,0.236842,NaN,0.064043,0.0,0.581810,0.176137,0.586966,0.235160,0.0,0.176
4,0014593270,732408,0.727665,0.283883,0.500550,0.066409,NaN,0.319265,0.0,0.399320,0.176137,0.391535,0.206400,0.0,0.176


## 9. Export new, versioned files and verify

In [ ]:
if WRI_MODE == "regenerate" and WRITE_OUTPUTS:
    # Existing Cell 18 export logic
    raw_path = OUT / "im3_eia_wri_13scores_raw.csv"
    norm_path = OUT / "im3_eia_wri_13scores_normalized_0to1.csv"
    feature_path = OUT / "wri_facility_13scores_raw.csv"
    for path in (raw_path, norm_path, feature_path):
        assert not path.exists(), f"Output exists; review it before overwriting: {path}"

    combined_raw.to_csv(raw_path, index=False)
    combined_norm.to_csv(norm_path, index=False)
    raw_features.to_csv(feature_path, index=False)

    for path in (raw_path, norm_path, feature_path):
        check = pd.read_csv(path, dtype={"id": "string"})
        assert len(check) == len(fac) and check["id"].is_unique
        print(f"Verified: {path} ({len(check):,} rows, {len(check.columns)} columns)")
    print("Complete. Review QA diagnostics before uploading to S3 or training.")


AssertionError: Output exists; review it before overwriting: /mnt/c/Users/attho/OneDrive/Desktop/College/ACM ATLAS/ATLAS/Data/processed/im3_eia_wri_13scores_raw.csv

## 10. How to use the outputs

- `im3_eia_wri_13scores_raw.csv`: full facility table plus raw 0–5 WRI scores; recommended archival and research baseline.
- `im3_eia_wri_13scores_normalized_0to1.csv`: full facility table plus 13 fixed-scale normalized scores; useful when the model expects inputs on comparable bounded scales.
- `wri_facility_13scores_raw.csv`: compact ID, watershed ID, and 13-score table.

**Before training:** inspect `gtd_score` missingness and constant/low-variance features. Do not impute absent groundwater data as zero. Consider watershed-aware evaluation if facilities share the same polygon. For any learned imputation or scaling, fit on training data only.

**If the join fails:** inspect coordinate order (lon, lat), CRS, WRI U.S. polygon coverage, invalid geometry, and exact-boundary points. Do not switch spatial predicates or resolve duplicates automatically without verifying the geography.


## in memory merge and validate for wri + im3 + eia

In [ ]:
# ATLAS + WRI merge using existing validated facility-level WRI features.
# Read-only: does not rerun the spatial join or write files.

atlas_path = ATLAS_CSV
wri_features_path = OUT / "wri_facility_13scores_raw.csv"

# Load datasets
assert atlas_path.is_file(), f"Missing ATLAS dataset: {atlas_path}"
assert wri_features_path.is_file(), f"Missing WRI features: {wri_features_path}"

atlas = pd.read_csv(atlas_path, dtype={"id": "string"})
wri_features = pd.read_csv(wri_features_path, dtype={"id": "string"})

# Validate facility IDs
for name, df in [("ATLAS", atlas), ("WRI", wri_features)]:
    assert "id" in df.columns, f"{name}: missing id column"
    assert df["id"].notna().all(), f"{name}: null facility IDs"
    assert df["id"].is_unique, f"{name}: duplicate facility IDs"

assert set(atlas["id"]) == set(wri_features["id"]), (
    "ATLAS and WRI facility IDs do not match"
)

# Validate WRI feature schema
assert set(SCORES).issubset(wri_features.columns), (
    f"Missing WRI indicators: {set(SCORES) - set(wri_features.columns)}"
)

overlap = (set(atlas.columns) & set(wri_features.columns)) - {"id"}
assert not overlap, f"Overlapping columns: {overlap}"

# Expected dimensions based on input datasets
expected_shape = (
    len(atlas),
    len(atlas.columns) + len(wri_features.columns) - 1,
)

# Merge raw WRI scores with canonical ATLAS dataset
atlas_wri_raw = atlas.merge(
    wri_features,
    on="id",
    how="left",
    validate="one_to_one",
    sort=False,
    indicator=True,
)

assert atlas_wri_raw["_merge"].eq("both").all(), (
    "Some facilities failed to match WRI features"
)

atlas_wri_raw = atlas_wri_raw.drop(columns="_merge")

assert atlas_wri_raw.shape == expected_shape

# Verify original ATLAS columns are unchanged
pd.testing.assert_frame_equal(
    atlas_wri_raw[atlas.columns],
    atlas,
)

# Normalize 13 WRI scores from 0–5 to 0–1
atlas_wri_norm = atlas_wri_raw.copy()

for score in SCORES:
    values = pd.to_numeric(atlas_wri_norm[score], errors="coerce")

    assert values.dropna().between(0, 5).all(), (
        f"{score}: values outside expected 0–5 range"
    )

    norm_name = score.removesuffix("_score") + "_norm"
    atlas_wri_norm[norm_name] = values / 5.0

atlas_wri_norm = atlas_wri_norm.drop(columns=SCORES)

# Final validation
assert atlas_wri_norm.shape == expected_shape
assert atlas_wri_raw["id"].is_unique
assert atlas_wri_norm["id"].is_unique

norm_cols = [
    score.removesuffix("_score") + "_norm"
    for score in SCORES
]

assert len(norm_cols) == 13
assert all(
    atlas_wri_norm[col].dropna().between(0, 1).all()
    for col in norm_cols
)

print("ATLAS + WRI merge validated.")
print("Raw shape:", atlas_wri_raw.shape)
print("Normalized shape:", atlas_wri_norm.shape)
print("Facility count:", len(atlas_wri_raw))
print("Normalized WRI indicators:", len(norm_cols))
print("No files written.")

ATLAS + WRI merge validated.
Raw shape: (1147, 63)
Normalized shape: (1147, 63)


## verify

In [13]:
# Compare the reconstructed merge with the existing CSV exports.
# Read-only: no files are overwritten.

saved_raw = pd.read_csv(
    OUT / "im3_atlas_wri_13scores_raw.csv"
)

saved_norm = pd.read_csv(
    OUT / "im3_atlas_wri_13scores_normalized_0to1.csv"
)

pd.testing.assert_frame_equal(
    atlas_wri_raw,
    saved_raw,
    check_dtype=False,
    check_exact=False,
    rtol=0,
    atol=1e-8,
)

pd.testing.assert_frame_equal(
    atlas_wri_norm,
    saved_norm,
    check_dtype=False,
    check_exact=False,
    rtol=0,
    atol=1e-8,
)

print("Existing ATLAS + WRI exports match the reconstructed merge.")

Existing ATLAS + WRI exports match the reconstructed merge.
